# Recommender Systems - Mini Challenge FS26

In this minichallenge we will explore a MovieLens dataset and implement several recommender systems and evaluation methods. Subsequently we will optimize these methods and compare the results. 

**Submission deadline:** Sunday of SW11 at 18:00 

## Guidelines for Implementation and Submission
- Code must be written in Python. The versions of all used packages must be given for reproducability.
- You may respond in English or German.
- We develop numerous algorithms ourselves. Unless explicitly stated otherwise, only the following libraries may be used in Python: numpy, matplotlib, seaborn, pandas. 
- Follow good coding practices and write modular, reusable code.
- The submitted solution must contain all codes and the results. No code may be outsourced.
- All pathes must be relative and just downloading your repo must be executable without modifications.
- Only fully running code is graded. The notebook must run sequential from start to end.
- During development, if computation time is too long for productive prototyping and debugging work, it is recommended to reduce the dataset to a fraction of its original. However, final results must be calculated on the full dataset. 
- All plots must be fully labeled (title, axes, labels, colorbar, etc.) so that the plot can be easily understood.
- Each plot must be accompanied by a brief discussion, which explains the plot and captures the key insights that become visible.
- Only fully labeled plots with an accompanying discussion will be assessed.
- The last commit in your fork of the repo before the submission deadline counts as the submission.
- Points will be deducted if you write inconsise (Denial of service will be punished) or if I read text not written for me but for the user of ChatGPT oir similar. 
- If you would like to submit and have the mini-challenge assessed, please send a short email to the subject expert (moritz.kirschmann@fhnw.ch) within 2 days after submission.
- Please do not delete, duplicate, or move the existing cells. This leads to problems during the correction. However, you may add as many additional cells as you like.

## Exercises

### Exercise 1 - A deep exploration of the dataset (17 points)
We will work with a subset of the MovieLens dataset. This subset is located under ``data/ml-latest-small``. Read the ``README.txt``carefully. 
Open the files. 

a) Describe the available data.

b) Find and fix bad data (e.g. duplicates, missing values, etc.).

Generate lists of

c) - Top 20 movies by average rating

d) - Top 20 movies by number of views

e) What is the range of the ratings? 

f) Which genre has be rated how many times?

g) How sparse is the User Rating Matrix?

Plot the following:

h) How many users have rated how many movies

i) Which rating is given how often over time with a time resolution of month 

j) Which rating is given how often per genre

k) The rating distributions of 10 random movies

l) The rating distributions of 3 movies that you have watched

m) How many users give which average rating

n) How often a movie was rated as a function of average rating

o) A heatmap of the User Item Matrix

p) A heatmap of the User Item Matrix for the 100 most rated movies for the 50 users with most ratings


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
import seaborn as sns

**a) Describe the available data**

As the provided summary states the dataset describes 5-star rating and free-text tagging activity from MovieLens, a movie recommendation service. It contains 100836 ratings and 3683 tag applications across 9742 movies made by 610 users between March 29, 1996 and September 24, 2018

In [ ]:
movies = pd.read_csv("./Data/ml-latest-small/movies.csv")
ratings = pd.read_csv("./Data/ml-latest-small/ratings.csv")
tags = pd.read_csv("./Data/ml-latest-small/tags.csv")

movies.info()
ratings.info()
tags.info()

The dataset consists of three main components: movies, ratings, and tags.

The `movies` dataset contains information on 9,742 movies each with unique `movieId`, including their titles and associated genres. Genres are provided as categorical descriptors and may include multiple labels per movie.

The `ratings` dataset contains 100,836 user ratings. Each entry represents a rating given by a specific user (`userId`) to a specific movie (`movieId`). Users can rate multiple movies, resulting in multiple entries per user. Ratings are given on a 5-star scale with half-star increments, ranging from 0.5 to 5.0. Additionally, each rating includes a timestamp indicating when the rating was submitted, measured in seconds since January 1, 1970 (UTC).

The `tags` dataset contains user-generated annotations for movies. Each entry links a user (`userId`) to a movie (`movieId`) and includes a tag, typically a short word or phrase describing some aspect of the movie. Similar to the ratings data, a timestamp is provided to indicate when the tag was created.

Overall, the dataset combines explicit feedback (ratings) with descriptive metadata (genres and tags).

**b) Find and fix bad data (e.g. duplicates, missing values, etc.).**

Beyond duplicates and missing values, I will also check for data consistency, validity, and structural issues.


In [ ]:
# 1) Ensure correct data types
ratings['userId'] = ratings['userId'].astype(int)
ratings['movieId'] = ratings['movieId'].astype(int)
movies['movieId'] = movies['movieId'].astype(int)
ratings['rating'] = ratings['rating'].astype(float)
tags['userId'] = tags['userId'].astype(int)
ratings['timestamp'] = pd.to_datetime(ratings['timestamp'], unit='s')
tags['timestamp'] = pd.to_datetime(tags['timestamp'], unit='s')

# 2) Delete any duplicates
movies = movies.drop_duplicates(subset=['movieId'])
ratings = ratings.drop_duplicates(subset=['userId', 'movieId'])

# 3) Check for any NaN
def nan_check(data):
    total_nans = data.isna().sum().sum()
    per_column = data.isna().sum()
    any_nans = data.isna().any().any()

    print("Total NaNs:", total_nans)
    print("\nNaNs per column:")
    print(per_column)
    print("\nAny NaNs present:", any_nans)

    return total_nans, per_column, any_nans

for name, data in [("movies", movies), ("ratings", ratings), ("tags", tags)]:
    print(f"\n--- {name} ---")
    nan_check(data)

# 4) Check for invalid ratings
invalid_ratings = ratings[(~ratings['rating'].between(0.5, 5.0)) | (ratings['rating'] % 0.5 != 0)]
print("Number of invalid ratings:", len(invalid_ratings))

# 5) Make sure all movieIds in ratings are in movies
if ratings['movieId'].isin(movies['movieId']).all():
    print("All movieIds in ratings exist in movies")
else:
    print("There are ratings for movies not present in the movies dataset")


I checked for common data quality issues and cleaned where necessary. I convertent columns to appropriate data types to ensure consistency, and made sure the timestamps are in a readable datetime format. Duplicate entries were then removed from the `movies` and `ratings` datasets based on their identifying columns.

Then I checked all three datasets (`movies`, `ratings`, and `tags`) for missing values with a reusable function. No missing values were found in any column. The ratings were also validated to ensure that they lie within the allowed range from 0.5 to 5.0 and follow the required half-star increments. No invalid ratings were detected. Finally, referential integrity was verified by confirming that all `movieId` values appearing in the `ratings` dataset also exist in the `movies` dataset.

Overall, the dataset seems clean, consistent, and suitable for building recommendation systems as expected from GroupLens.

**c) - Top 20 movies by average rating**

In [ ]:
# Make a dataframe with title and average ratings
movieId_avrating = ratings.groupby('movieId')['rating'].mean().reset_index()
movies_with_ratings = movies.merge(movieId_avrating, on='movieId', how='inner')
print(movies_with_ratings.sort_values(by='rating', ascending=False).head(20)) # print top 20

I made a list (not a literal Python list) of the top 20 movies based on their average rating. It was generated by grouping the ratings by movieId, computing the mean rating, and sorting the results in descending order. It is clear that the top spots are dominated by movies with perfect scores, which likely have only very few reviews. In a recommender system, a more robust approach would be to introduce a damping or weighting mechanism to penalise entries with fewer reviews. However, I assume this is not required for this task and will consider it later when building the actual recommendation system.

**d) - Top 20 movies by number of views**

In [ ]:
# Make a dataframe with title and amount of ratings
rating_amount = ratings.groupby('movieId')['rating'].count().reset_index(name='num_ratings')
movies_with_rating_amount = movies.merge(rating_amount, on='movieId', how='inner')
print(movies_with_rating_amount.sort_values(by='num_ratings', ascending=False).head(20)) # print top 20

I presumed that “top 20 by views” refers to the top 20 movies by the number of reviews in the ratings dataset. I used the same technique as for the top 20 list by average rating. It becomes clear that the most viewed movies are not the same as the top 20 highest-rated movies. This is likely because a larger number of reviewers introduces more variation in the scores due to users’ individual preferences. This indicates that the number of ratings should be taken into account when designing a robust recommender system.

**e) What is the range of the ratings?**

In [ ]:
print("Min rating:", ratings['rating'].min()) # As expected the worst rating is the lowest possible rating of 0.5
print("Max rating:", ratings['rating'].max()) # As expected the best rating is the highest possible rating of 5.0

**f) Which genre has be rated how many times?**

In [ ]:
# merge ratings with movies to get genres per rating
ratings_with_genres = ratings.merge(movies[['movieId', 'genres']], on='movieId')

# split genres and explode into separate rows
genres_exploded = ratings_with_genres['genres'].str.split('|').explode()

# count how often each genre was rated
genre_counts = genres_exploded.value_counts()

print(genre_counts) # Shows how many times has each genre been rated

**g) How sparse is the User Rating Matrix?**

Sparsity describes how many entries in the user–item matrix are missing (i.e., movies that users have not rated). It can be calculated as:

$$ sprasity = 1 - \frac{number~of~ratings}{number~of~users~\times~number~of~movies}$$

In [ ]:
n_users = ratings['userId'].nunique()
n_movies = ratings['movieId'].nunique()
n_ratings = len(ratings)

sparsity = 1 - (n_ratings / (n_users * n_movies))

print("Number of users:", n_users)
print("Number of movies:", n_movies)
print("Number of ratings:", n_ratings)
print("Sparsity:", sparsity)

I calculated the sparsity of the user–rating matrix and found it to be approximately 0.983. This means that about 98.3% of all possible user–movie interactions are missing, and only a very small fraction of ratings is actually observed. This confirms that the dataset is highly sparse, which is typical for recommender systems and makes the task of generating accurate recommendations more challenging.

**h) How many users have rated how many movies**

In [ ]:
# Count how many movies each user rated
ratings_per_user = ratings.groupby('userId')['movieId'].count()

# plot distribution
plt.figure(figsize=(10, 6))
plt.hist(ratings_per_user, bins=30)

plt.title("Distribution of Number of Ratings per User")
plt.xlabel("Number of Movies Rated")
plt.ylabel("Number of Users")

plt.tight_layout()
plt.show()

As I already intreperted by the calculated sparsity, the left-skewed histogram shows that most users have rated only a small number of movies, while a few users have rated a much larger number. This highlights the uneven distribution of user activity in the dataset.

**i) Which rating is given how often over time with a time resolution of month**

In [ ]:
# make sure timestamp is datetime
ratings['timestamp'] = pd.to_datetime(ratings['timestamp'], unit='s')

# extract month
ratings['month'] = ratings['timestamp'].dt.month

# count ratings per month and rating
month_rating = (
    ratings
    .groupby(['month', 'rating'])
    .size()
    .unstack(fill_value=0)
)

# normalize per month
month_rating_norm = month_rating.div(month_rating.sum(axis=1), axis=0)

# sort columns so colors match ratings correctly
month_rating_norm = month_rating_norm.sort_index(axis=1)

# define rating colors more or less based on color spectrum
rating_colors = {
    0.5: 'purple',
    1.0: 'pink',
    1.5: 'blue',
    2.0: 'turquoise',
    2.5: 'cyan',
    3.0: 'green',
    3.5: 'yellow',
    4.0: 'orange',
    4.5: 'red',
    5.0: 'brown'
}

# apply fixed colors
colors = [rating_colors[r] for r in month_rating_norm.columns]

# plot
month_rating_norm.plot(kind='bar', stacked=True, figsize=(12, 7), color=colors)

plt.title("Normalized rating distribution per month (All Years Aggregated)")
plt.xlabel("Month")
plt.ylabel("Proportion of Ratings")

plt.xticks(
    ticks=range(12),
    labels=['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
            'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'],
    rotation=0)

plt.legend(title="Rating", loc='center left', bbox_to_anchor=(1, 0.5))
plt.tight_layout()
plt.show()

My key takeaway from this plot is that higher ratings (around 4.0 – 5.0) make the majority of ratings, while lower ratings tend to get rarer the lower they are. The time of the year does not really seem to have a big impact on the user's rating score. This indicates a positive bias in the dataset, where users are more likely to rate movies they enjoyed.

**j) Which rating is given how often per genre**

In [ ]:
# merge and explode genres
ratings_genres = ratings.merge(movies[['movieId', 'genres']], on='movieId')
ratings_genres = ratings_genres.assign(
    genre=ratings_genres['genres'].str.split('|')
).explode('genre')


ratings_genres = ratings_genres[ratings_genres['genre'] != '(no genres listed)'] # remove placeholer

# count ratings per genre
genre_rating = (
    ratings_genres
    .groupby(['genre', 'rating'])
    .size()
    .unstack(fill_value=0)
)

# normalize for better visualization
genre_rating_norm = genre_rating.div(genre_rating.sum(axis=1), axis=0)

# same color sheme as bafore
rating_colors = {
    0.5: 'purple',
    1.0: 'pink',
    1.5: 'blue',
    2.0: 'turquoise',
    2.5: 'cyan',
    3.0: 'green',
    3.5: 'yellow',
    4.0: 'orange',
    4.5: 'red',
    5.0: 'brown'}

# colors in correct order
colors = [rating_colors[r] for r in genre_rating_norm.columns]

# plot
genre_rating_norm.plot(kind='bar', stacked=True, figsize=(14, 8), color=colors)

plt.title("Normalized Rating Distribution per Genre")
plt.xlabel("Genre")
plt.ylabel("Proportion of Ratings")

plt.xticks(rotation=45)
plt.legend(title="Rating", loc='center left', bbox_to_anchor=(1, 0.5))

plt.tight_layout()
plt.show()

As already observed before, higher ratings occur more frequently than lower ones. This pattern is consistent across all genres, where ratings around 4.0–5.0 dominate, while very low ratings are relatively rare. This suggests a general positive bias in user ratings, independent of month and genre.

**k) The rating distributions of 10 random movies**

In [ ]:
# randomly select 10 movies each run
random_movies = movies.sample(n=10, random_state=None)  # change random_state to int like 1 ro remove randomness

# merge with ratings
ratings_subset = ratings.merge(random_movies[['movieId', 'title']], on='movieId')

# plot distributions
plt.figure(figsize=(12, 6))

for title, group in ratings_subset.groupby('title'):
    counts = group['rating'].value_counts().sort_index()
    plt.plot(counts.index, counts.values, marker='o', label=title)

plt.title("Rating Distributions of 10 Random Movies")
plt.xlabel("Rating")
plt.ylabel("Number of Ratings")
plt.gca().yaxis.set_major_locator(MaxNLocator(integer=True))

plt.legend(bbox_to_anchor=(1, 1))
plt.tight_layout()
plt.show() # Cool plot, not much to say

**l) The rating distributions of 3 movies that you have watched**

In [ ]:
# select your movies (partial match to be safe)
selected_movies = movies[movies['title'].str.contains('Toy Story', case=False)] # This would not have worked if Toy Story 4 was in the data

# merge with ratings
ratings_subset = ratings.merge(selected_movies[['movieId', 'title']], on='movieId')

# plot
plt.figure(figsize=(10, 6))

for title, group in ratings_subset.groupby('title'):
    counts = group['rating'].value_counts().sort_index()
    plt.plot(counts.index, counts.values, marker='o', label=title)

plt.title("Rating distributions of selected movies")
plt.xlabel("Rating")
plt.ylabel("Number of Ratings")

plt.legend()
plt.tight_layout()
plt.show()

I consider the Toy Story trilogy to be one of the most significant animated film series. The rating distributions reflect this, as all three movies receive predominantly positive reviews, with ratings concentrated around higher values. However, the number of ratings decreases for the later installments, suggesting that fewer users engaged with the newer movies. In my personal opinion, the movies improved with each sequel, culminating in Toy Story 3 as a strong conclusion to the series. Toy Story 4 is not included in the dataset, but I found it to be a disappointing addition. It felt unnecessary and, in my view, did not contribute meaningfully beyond commercial reasons.

**m) How many users give which average rating**

In [ ]:
# compute average rating per user
user_avg = ratings.groupby('userId')['rating'].mean()

# plot
plt.figure(figsize=(10, 6))
plt.hist(user_avg, bins=20)

plt.title("Distribution of Users' Average Ratings")
plt.xlabel("Average Rating per User")
plt.ylabel("Number of Users")

plt.tight_layout()
plt.show()

I calculated the average rating for each user and visualised the distribution using a histogram. As observed earlier, most users tend to give relatively high ratings, with the majority of average ratings lying above approximately 3.25. This confirms that users generally rate movies positively, leading to a noticeable skew towards higher average ratings.

**n) How often a movie was rated as a function of average rating**

In [ ]:
# compute average rating and number of ratings per movie
movie_stats = ratings.groupby('movieId')['rating'].agg(['mean', 'count']).reset_index()

# plot
plt.figure(figsize=(10, 6))
plt.scatter(movie_stats['mean'], movie_stats['count'], alpha=0.5)

plt.title("Number of Ratings vs Average Rating per Movie")
plt.xlabel("Average Rating")
plt.ylabel("Number of Ratings")

plt.tight_layout()
plt.show()

The scatter plot shows the relationship between the average rating of a movie and how often it was rated. Most movies have relatively few ratings and are spread across a wide range of average scores. However, movies with a large number of ratings tend to cluster around average ratings between roughly 3.0 and 4.5. Very highly rated movies (close to 5.0) usually have only a small number of ratings, which suggests that these scores are less reliable. I had not quite noticed this before, as I was mainly looking at how often a rating of 5.0 occurred within a genre or month, rather than considering how many total ratings those movies had. Overall, this indicates that popular movies tend to receive more moderate but stable ratings, while extreme ratings are often based on fewer user opinions.

**o) A heatmap of the User Item Matrix**

**p) A heatmap of the User Item Matrix for the 100 most rated movies for the 50 users with most ratings**

In [ ]:
# o
# create user matrix
user_item = ratings.pivot(index='userId', columns='movieId', values='rating')

user_item_sample = user_item.sample(n=100, random_state=None) # sampling is necessary for visibility

# plot heatmap
plt.figure(figsize=(12, 6))
sns.heatmap(user_item_sample, cmap='viridis', cbar=True)

plt.title("User–Item Matrix (Sample of Users)")
plt.xlabel("Movie ID")
plt.ylabel("User ID")

plt.tight_layout()
plt.show()

In [ ]:
# p
# top 50 users by number of ratings
top_users = ratings['userId'].value_counts().head(50).index

# top 100 movies by number of ratings
top_movies = ratings['movieId'].value_counts().head(100).index

# filter data
filtered = ratings[
    (ratings['userId'].isin(top_users)) &
    (ratings['movieId'].isin(top_movies))
]

# create matrix
user_item_filtered = filtered.pivot(index='userId', columns='movieId', values='rating')

# plot heatmap
plt.figure(figsize=(14, 8))
sns.heatmap(user_item_filtered, cmap='viridis', cbar=True)

plt.title("User–Item Matrix (Top 50 users & top 100 movies)")
plt.xlabel("Movie ID")
plt.ylabel("User ID")

plt.tight_layout()
plt.show()

The full user–item matrix appears very empty, even after sampling, which is due to the high sparsity of the dataset we calculated in a previous cell. Since most users have only rated a small subset of all available movies, the majority of entries in the matrix are missing. In contrast, the heatmap for the most active users and most frequently rated movies appears much denser and reveals clearer patterns. This demonstrates how sparsity strongly affects the visibility of structure in the data.

### Exercise 2 - Building a baseline RS (7 points)
In this exercise we will build a baseline RS and functions to calculate fundamental performance metrics. 

Build the following baseline RS to predict Top-N (default N=20):
1. In reference to the book *Collaborative Filtering Recommender Systems by Michael D. Ekstrand, John T. Riedl and Joseph A. Konstan* (p. 91ff) implement the baseline predictor $$ b_{u,i}= \mu +b_u +b_i $$

Elements:

* $b_{u,i}$: predicted rating for user $u$ and movie $i$
* $\mu$: global average rating across all ratings
* $b_u$: user bias / user offset
  * captures whether user $u$ generally rates higher or lower than average
* $b_i$: item bias / movie offset
  * captures whether movie $i$ is generally rated higher or lower than expected



Regularized user offset: $$ b_u = \frac{1}{|I_u| + \beta_u} \sum_{i \in I_u} (r_{u,i} - \mu) $$
* $r_{u,i}$: actual rating given by user $u$ to movie $i$
* $I_u$: set of movies rated by user $u$
* $|I_u|$: number of movies rated by user $u$
* $\beta_u$: damping factor for user bias
    * reduces extreme user offsets when a user has few ratings

Regularized item average offset: $$ b_i = \frac{1}{|U_i| + \beta_i} \sum_{u \in U_i} (r_{u,i} - b_u - \mu) . $$
* $U_i$: set of users who rated movie $i$
* $|U_i|$: number of users who rated movie $i$
* $\beta_i$: damping factor for item bias
    * reduces extreme movie offsets when a movie has few ratings

Build a recommender system upon this baseline predictor. Set the default damping factors $\beta_u$ and $\beta_i$ both to 20.

In simple terms:

* $\mu$ gives the general rating level.
* $b_u$ adjusts for users who rate stricter or more generously.
* $b_i$ adjusts for movies that are generally liked or disliked.
* The damping factors prevent unreliable offsets from users or movies with very few ratings.

2. Build a RS that recommends based on *random* recommendations.  

Output the recommendations for three example users (Ids 1, 3 and 7) and the default parameters. Give the titles of the recommended movies and their predicted scores not just their Ids.

In [52]:
# first formula: b_u = 1/(|I_u| + beta_u) * sum(r_ui - mu)
def compute_user_bias(ratings, mu, beta_u=20):
    # number of ratings per user |I_u|
    user_counts = ratings.groupby('userId').size()

    # sum of (r_ui - mu) per user
    user_deviation = ratings.groupby('userId')['rating'].apply(lambda x: (x - mu).sum())

    # formula in code
    b_u = user_deviation / (user_counts + beta_u)

    return b_u

# second formula: b_i = 1 / (|U_i| + beta_i) * sum(r_ui - b_u - mu)
def compute_item_bias(ratings, mu, b_u, beta_i=20):
    # map user bias to each rating
    ratings_with_bu = ratings.copy()
    ratings_with_bu['b_u'] = ratings_with_bu['userId'].map(b_u)

    # number of ratings per item |U_i|
    item_counts = ratings_with_bu.groupby('movieId').size()

    # sum of (r_ui - b_u - mu) per item
    item_deviation = ratings_with_bu.groupby('movieId').apply(
        lambda x: (x['rating'] - x['b_u'] - mu).sum()
    )

    # formula in code
    b_i = item_deviation / (item_counts + beta_i)

    return b_i


# third formula: mu + b_u + b_i
def predict_baseline(user_id, movie_id, mu, b_u, b_i):
    # get user bias (default 0 if user not seen)
    bu = b_u.get(user_id, 0)

    # get item bias (default 0 if movie not seen)
    bi = b_i.get(movie_id, 0)

    # baseline prediction rating
    return mu + bu + bi

# THE BASELINE PREDICTOR
def recommend_baseline(user_id, ratings, movies, beta_u=20, beta_i=20, n=20):
    mu = ratings['rating'].mean()
    b_u = compute_user_bias(ratings, mu, beta_u)
    b_i = compute_item_bias(ratings, mu, b_u, beta_i)

    rated_movies = ratings.loc[ratings['userId'] == user_id, 'movieId']
    candidates = movies[~movies['movieId'].isin(rated_movies)].copy()

    candidates['predicted_score'] = candidates['movieId'].apply(
        lambda movie_id: predict_baseline(user_id, movie_id, mu, b_u, b_i)
    )

    recs = (
        candidates
        .sort_values(by='predicted_score', ascending=False)
        .head(n)[['title', 'predicted_score']]
        .copy()
    )

    recs.insert(0, "Rank", range(1, len(recs) + 1))
    recs['predicted_score'] = recs['predicted_score'].round(3) # more exact predicted score
    # recs['predicted_score'] = ((recs['predicted_score'] * 2).round() / 2).map(lambda x: f"{x:.1f}")  # rounds to MovieLens 0.5 rating steps

    return recs

# THE RANDOM PREDICTOR
def recommend_random(user_id, ratings, movies, n=20):
    # movies the user has already rated
    rated_movies = ratings.loc[ratings['userId'] == user_id, 'movieId']

    # candidate movies = movies not yet rated by the user
    candidates = movies[~movies['movieId'].isin(rated_movies)].copy()

    # randomly select N movies
    recommendations = (
        candidates
        .sample(n=n, random_state=None)
        [['movieId', 'title']]
    )

    return recommendations

In [53]:
print("Recommendations for user 1:",
      recommend_baseline(1, ratings, movies),
      recommend_random(1, ratings, movies))

print("Recommendations for user 3:",
      recommend_baseline(3, ratings, movies),
      recommend_random(3, ratings, movies))

print("Recommendations for user 7:",recommend_baseline(7, ratings, movies),
      recommend_random(7, ratings, movies))

Recommendations for user 1:       Rank                                              title  predicted_score
277      1                   Shawshank Redemption, The (1994)            5.033
659      2                              Godfather, The (1972)            4.881
602      3  Dr. Strangelove or: How I Learned to Stop Worr...            4.877
906      4                          Lawrence of Arabia (1962)            4.849
686      5                                 Rear Window (1954)            4.829
922      6                     Godfather: Part II, The (1974)            4.828
694      7                                  Casablanca (1942)            4.815
6710     8                            Dark Knight, The (2008)            4.814
4909     9       Eternal Sunshine of the Spotless Mind (2004)            4.792
6315    10                               Departed, The (2006)            4.790
896     11             One Flew Over the Cuckoo's Nest (1975)            4.787
3622    12  Amelie (Fabu

### Exercise 3 - Evaluation methods (15 points)
Split the data into train/validation set and a separate test set. This test set shall contain the first 20% of the users and shall not be used at all before exercise 10. With the remaining 80% do the following: 
Implement a function to partition your dataset for an offline evaluation based on holding out of random users with 5x cross validation with a 80/20 train/validation split. Within the validation set implement a masking with *all but n* approach. 
See page 2942 of https://jmlr.csail.mit.edu/papers/volume10/gunawardana09a/gunawardana09a.pdf for details on this approach. 

Choose the number of masked items n reasonably and explain your considerations.

Implement functions to calculate the following metrics:
- *Mean Absolute Error (MAE)* 
- *Root Mean Square Error (RMSE)*
- *Precision@N* with default $N=5$ and relevance threshold 4.0 stars.
- *Recall@N* with default $N=5$ and relevance threshold 4.0 stars.
- *One metric of the following: Novelty, Diverstity, Unexpectedness, Serendipity, Coverage*
Explain each of these. How does the relevance threshold influence the metrics? How would you choose this parameter?

Note: For *precision@N* and *Recall@N* use the definitions from https://medium.com/@m_n_malaeb/recall-and-precision-at-k-for-recommender-systems-618483226c54 with one exception: In case of the denominator being zero, set the metric to 0. 

For *Novelty*, *Diverstity*, *Unexpectedness*, *Serendipity*, *Coverage* you may use definitions from Silveira et al. https://link.springer.com/article/10.1007/s13042-017-0762-9 

### Exercise 4 - Optimize hyperparameters of baseline RS (6 points)
Optimize the hyperparameters $\beta_u$ and $\beta_i$ for the baseline RS from exercise 2 based on the RMSE metric. To save computation time find a reasonable maximum value for the betas. Explain your approach and your solution.
Plot the MAE, RMSE, Precision@N, Recall@N as functions of the betas.

Which metric would you use for hyperparameter tuning? Explain your decision.

### Exercise 5 - Collaborative filtering; item-based and user-based (16 points)
In this exersise we will build 8 different collaborative-filtering RS based on nearest neighbour technique, both in terms of item and user. 

Implement:
1. a RS based on the $K$ most similar items (K nearest neighbours). Similarity shall be calculated based on *cosine similarity*. 
2. a RS based on the $K$ most similar items (K nearest neighbours). Similarity shall be calculated based on *Pearson Correlation Coefficienct*. 
3. a RS based on the $K$ most similar users (K nearest neighbours). Similarity shall be calculated based on *cosine similarity*. 
4. a RS based on the $K$ most similar users (K nearest neighbours). Similarity shall be calculated based on *Pearson Correlation Coefficienct*. 

Make for each of the models above 2 variants. One with user-wise zero centering and one without.

Each should have a default $K$ of 20.

Explain how you handle NaN values in the user rating matrix when computing similarities? What other preparations are useful such as normalization and mean centering?

Describe the two similarity metrics.

Show the top 20 recommended items for user ids 3, 5 and 7.


### Exercise 6 - Optimize hyperparameter $K$ (6 points)
Optimize the hyperparameter $K$ for all RS from the prior exercise optimizing for minimal RMSE. 
For each RS plot RMSE, Precision@N and Recall@N as a function of $K$. 

Compare the results of these four RS on the 3 example users. Do the results match your expectation? Describe.

### Exercise 7 - Model-based RS: SVD (10 points)
In this exercise we will use the unsupervised method *singular value decomposition (SVD)* from the python package *surprise* (https://surpriselib.com, documentation https://surprise.readthedocs.io/en/stable/matrix_factorization.html). SVD can compress much of the information of a matrix in few components.  

a)Run the SVD RS and show the results on the three example users from exercise 2. Explain how this algorithm works.

Note: A very good general introduction to SVD is this youtube video series starting with https://www.youtube.com/watch?v=gXbThCXjZFM&t=337s . See *Collaborative filtering recommender systems* by Ekstrand et al. *Mining of massive datasets* by Leskovec, Kapitel 11 (2020) and, *Recommender systems: The textbook*, by Aggarwal, chapter 3

b) We explore now what latent factors SVD has learned. Generate an interactive 2D UMAP plot of all latent movie factors. Make versions of this plot with fixed values for *n_neighbours* and *min_dist* each time coloring the points by different parameters: year, genre, numbers of genres, average rating and number of ratings to explore the resulting plot. Can you see any patterns?

Background: UMAP is a method for dimensionality reduction. Dimensionality reduction is usefull to respresent high dimensional data sets in less dimensions with goal to allow interpretable 2D/3D visualization. See for the documentation of the python package https://umap-learn.readthedocs.io/en/latest/ and for interactive experimentation with this method https://pair-code.github.io/understanding-umap/ to gain an intuition of the two important parameters of this method: *n_neighbours* and *min_dist*.

**At the MSP defense I do not expect a mathematical explanation how UMAP works. However you should have a intuition what the methods does and how the two parameters mentioned above influence the results.**


### Exercise 8 - Optimize hyperparameter $k$ or `n_factors` (4 points)
Optimize the hyperparameter, representing the number of greatest SVD components used for the truncated reconstruction of the user item matrix, to minimize RMSE.
Plot RMSE, Precision@N and Recall@N as a function of this hyperparameter. Finally output all performance metrics from exercise 3 for the optimal $k$ value.

### Exercise 9 - Everything goes (30 points)
In this exercise you can explore different methods of RS. You are not limited what methods you apply. You can try to improve the methods from the earlier exercises by modifiying them or generating ensemble or hybrid RS. Also you could train deep neural networks, use NLP methods, use the available links to imdb available in the dataset to further enrich the dataset or find an obscure method by someone else on Github. 
Document what your inspirations and sources are and describe the method conceptually. 

**Build and optimize in total *three* different methods. The last one has the additional requirement that it should increase the diversity of the recommendations in order to minimize filter bubbles.**

**Important: If you use the work of someone else you must be able to explain the method conceptually during the defense MSP.** 

Output the performance metrics of exercise 3. 

### Exercise 10 - Compare all RS that you build in this challenge (8 points)
a) Compile a table with the performance metrics of exercise 3 for all RS from this MC (Make sure to include the baseline RS and random RS) on the test set defined in exercise 3. Also generate comparative plots. Discuss.

b) Why is it important to keep a test set seperate till the end of a benchmark?

**Read the Guidelines for Implementation and Submission one more time.**